## 实验5：栈对象和堆对象

本实验通过构造和析构输出，对比自动存储期对象与动态分配对象的生命周期。重点不是记住“栈快、堆慢”，而是明确：对象由谁创建、由谁持有、何时销毁，以及忘记释放会产生什么后果。

> “栈对象”和“堆对象”是常见的工程简称。更准确的 C++ 术语分别是**自动存储期对象**和**动态存储期对象**；语言标准并不要求自动对象必须由某种特定的物理栈实现。

### 实验代码

In [1]:
#include <iostream>

In [2]:
class User {
public:
    User() {
        std::cout
            << "constructor\n";
    }

    ~User() {
        std::cout
            << "destructor\n";
    }
};


{
    std::cout
        << "stack object\n";

    User stack_user;

    std::cout
        << "heap object\n";

    User* heap_user = new User;

    std::cout
        << "delete heap object\n";

    delete heap_user;

    std::cout
        << "end\n";
}

stack object
constructor
heap object
constructor
delete heap object
destructor
end
destructor


### 自动存储期对象（常称栈对象）
```C++
User user;
```

生命周期：
```
进入作用域

 ↓

构造

 ↓

使用

 ↓

离开作用域

 ↓

自动析构
```
所以，自动对象的销毁与词法作用域绑定。即使作用域通过正常的提前 `return` 或异常展开离开，已经构造成功的局部对象也会按规则自动析构。

同一作用域中有多个自动对象时，它们通常按照构造顺序的逆序析构。这种确定性是 RAII 能够可靠释放资源的基础。

### 动态存储期对象（常称堆对象）
```C++
User* user = new User;
```

发生：
```
分配内存
      ↓
构造
      ↓
返回指针
```

销毁：
```C++
delete user;
```
发生：
```
析构
    ↓
释放内存
```

这里需要区分两个对象：

- `heap_user` 是当前作用域中的指针变量，本身具有自动存储期。
- `new User` 创建的 `User` 对象具有动态存储期，它不会因为指针变量离开作用域而自动销毁。

如果忘记 `delete`，动态对象及其占用的资源将无法通过这个指针释放，形成**内存泄漏**。执行 `delete heap_user` 后，动态对象已经销毁，但 `heap_user` 仍保存旧地址，成为**悬空指针**，不能再解引用或重复 `delete`。

`new`/`delete`、`new[]`/`delete[]` 必须正确配对；混用或重复释放都会导致未定义行为。

### 观察输出顺序

```text
stack object
constructor
heap object
constructor
delete heap object
destructor
end
destructor
```

第一个 `destructor` 由显式 `delete heap_user` 触发，销毁动态对象；最后一个 `destructor` 在外层作用域结束时自动触发，销毁 `stack_user`。这说明指针变量离开作用域和它所指动态对象被销毁，是两件不同的事。

### Ownership：谁负责释放？

| 表达式 | 创建方式 | 谁负责销毁 | 销毁时机 |
| --- | --- | --- | --- |
| `User stack_user;` | 自动创建 | 当前作用域 | 离开作用域时自动析构 |
| `User* heap_user = new User;` | 动态分配 | 获得所有权的代码 | 必须显式 `delete`，否则泄漏 |

裸指针 `User*` 本身无法表达它究竟是 owner（负责释放）还是 borrower（仅临时访问）。当代码出现提前返回、多个分支或异常时，手动保证恰好执行一次 `delete` 会迅速变得困难。

### 不要习惯直接写 `new/delete`
现在我们的实验代码需要理解它，所以我们手写了：
```C++
new
delete
```

但是现代 C++ 的原则是：**不要让裸 `new/delete` 在业务代码中传播**。优先让对象直接具有自动存储期；确实需要动态生命周期时，使用能够表达所有权的 RAII 类型。

后续实验会逐渐使用：
```C++
std::unique_ptr<User>
```
它会在离开作用域时自动销毁所拥有的对象，即使中途发生异常，也不需要手写 `delete`。更核心的思想是 **RAII**：把资源的获取与对象构造绑定，把资源释放与对象析构绑定。

因此，这里学习 `new/delete` 是为了理解动态分配、析构与 ownership 的底层关系，而不是把它们作为推荐写法。

### 实验结论

- 自动对象的生命周期由作用域管理，通常是最简单、最安全的选择。
- 动态对象的生命周期独立于保存地址的裸指针，必须有明确 owner。
- `delete` 同时执行析构函数并释放动态存储；之后原指针不可再使用。
- 现代 C++ 应使用自动对象、容器和智能指针，把资源释放交给 RAII。
- 这套 ownership/lifetime 思维会直接延伸到 C ABI 的 opaque handle 和 Kotlin/Native 的资源封装。